# 🧠 DriverPulse — Model Training Pipeline

**Three models trained on cleaned ward-level data:**
1. **Ward Risk Classifier** — High/Med/Low risk labeling with Random Forest
2. **Acceptance Predictor** — Regression on `driver_quote_acceptance_rate`
3. **Driver Behavior Clustering** — KMeans (k=4) with business labels

All models, scalers, and labels are saved to `models/` as `.pkl`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
    silhouette_score
)
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

os.makedirs('models', exist_ok=True)
print('Libraries loaded ✅')

In [ ]:
df = pd.read_csv('data/cleaned.csv')
print(f'Shape: {df.shape}')
df.head()

---
## Model 1: Ward Risk Classifier

**Strategy:**
1. Create risk labels using percentile-based thresholds on a composite risk score built from `driver_cancellation_rate`, `driver_quote_acceptance_rate`, and `conversion_rate`.
2. Train a Random Forest classifier to predict risk labels.
3. Evaluate with accuracy, F1, and confusion matrix.
4. Explain with SHAP.

### 1.1 Create Risk Labels (Percentile-Based)

In [ ]:
# Composite risk score:
#   High driver_cancellation_rate → HIGH risk
#   Low driver_quote_acceptance_rate → HIGH risk
#   Low conversion_rate → HIGH risk
# We normalize each and combine into a single score.

from sklearn.preprocessing import MinMaxScaler

risk_features = ['driver_cancellation_rate', 'driver_quote_acceptance_rate', 'conversion_rate']
risk_df = df[risk_features].copy()

# Higher cancel → higher risk; lower acceptance/conversion → higher risk
risk_df['cancel_norm'] = MinMaxScaler().fit_transform(risk_df[['driver_cancellation_rate']])
risk_df['accept_norm'] = 1 - MinMaxScaler().fit_transform(risk_df[['driver_quote_acceptance_rate']])
risk_df['conv_norm'] = 1 - MinMaxScaler().fit_transform(risk_df[['conversion_rate']])

risk_df['risk_score'] = (risk_df['cancel_norm'] + risk_df['accept_norm'] + risk_df['conv_norm']) / 3

# Percentile-based labeling
p33 = risk_df['risk_score'].quantile(0.33)
p66 = risk_df['risk_score'].quantile(0.66)

def label_risk(score):
    if score <= p33:
        return 'Low'
    elif score <= p66:
        return 'Medium'
    else:
        return 'High'

df['risk_label'] = risk_df['risk_score'].apply(label_risk)

print('Risk label distribution:')
print(df['risk_label'].value_counts())
print(f'\nPercentile thresholds: p33={p33:.4f}, p66={p66:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
df['risk_label'].value_counts().plot(kind='bar', color=[colors[l] for l in df['risk_label'].value_counts().index], ax=ax, edgecolor='white')
ax.set_title('Ward Risk Label Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Risk Level')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 1.2 Train Random Forest Classifier

In [ ]:
# Features for classification
clf_features = [
    'driver_cancellation_rate', 'driver_quote_acceptance_rate', 'conversion_rate',
    'booking_cancellation_rate', 'user_cancellation_rate',
    'earnings_per_km', 'avg_distance', 'avg_fare',
    'supply_gap', 'reliability_score'
]

X_clf = df[clf_features].copy()
le = LabelEncoder()
y_clf = le.fit_transform(df['risk_label'])  # High=0, Low=1, Medium=2

print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f'Train: {X_train_c.shape}, Test: {X_test_c.shape}')

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_split=5,
    random_state=42, class_weight='balanced', n_jobs=-1
)
rf_clf.fit(X_train_c, y_train_c)

y_pred_c = rf_clf.predict(X_test_c)

print('=== Ward Risk Classifier Results ===')
print(f'Accuracy:  {accuracy_score(y_test_c, y_pred_c):.4f}')
print(f'F1 (macro): {f1_score(y_test_c, y_pred_c, average="macro"):.4f}')
print(f'F1 (weighted): {f1_score(y_test_c, y_pred_c, average="weighted"):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test_c, y_pred_c, target_names=le.classes_))

In [ ]:
# Cross-validation
cv_scores = cross_val_score(rf_clf, X_clf, y_clf, cv=5, scoring='f1_macro')
print(f'5-Fold CV F1 (macro): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred_c)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_,
    ax=ax, linewidths=0.5
)
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix — Ward Risk Classifier', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.3 SHAP — Classifier Explainability

In [ ]:
explainer_clf = shap.TreeExplainer(rf_clf)
shap_values_clf = explainer_clf.shap_values(X_test_c)

# For multi-class, shap_values is a list of arrays (one per class)
# Show summary for each class
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for i, cls_name in enumerate(le.classes_):
    plt.sca(axes[i])
    shap.summary_plot(
        shap_values_clf[i] if isinstance(shap_values_clf, list) else shap_values_clf[:, :, i],
        X_test_c,
        plot_type='bar',
        show=False,
        max_display=10
    )
    axes[i].set_title(f'SHAP — {cls_name} Risk', fontsize=12, fontweight='bold')

plt.suptitle('SHAP Feature Importance — Ward Risk Classifier', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Overall SHAP beeswarm (for the "High" risk class)
high_idx = list(le.classes_).index('High')
plt.figure(figsize=(12, 7))
shap.summary_plot(
    shap_values_clf[high_idx] if isinstance(shap_values_clf, list) else shap_values_clf[:, :, high_idx],
    X_test_c,
    show=False,
    max_display=10
)
plt.title('SHAP Beeswarm — High Risk Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Save classifier
joblib.dump(rf_clf, 'models/ward_risk_classifier.pkl')
joblib.dump(le, 'models/risk_label_encoder.pkl')
print('✅ Saved: models/ward_risk_classifier.pkl')
print('✅ Saved: models/risk_label_encoder.pkl')

---
## Model 2: Acceptance Predictor (Regression)

Predict `driver_quote_acceptance_rate` using:
- `earnings_per_km`, `avg_distance`, `avg_fare`, `supply_gap`

**Evaluation:** RMSE, MAE, R²  
**Explainability:** SHAP

In [ ]:
reg_features = ['earnings_per_km', 'avg_distance', 'avg_fare', 'supply_gap']
target_reg = 'driver_quote_acceptance_rate'

X_reg = df[reg_features].copy()
y_reg = df[target_reg].copy()

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f'Train: {X_train_r.shape}, Test: {X_test_r.shape}')

In [ ]:
rf_reg = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_split=5,
    random_state=42, n_jobs=-1
)
rf_reg.fit(X_train_r, y_train_r)

y_pred_r = rf_reg.predict(X_test_r)

rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
mae = mean_absolute_error(y_test_r, y_pred_r)
r2 = r2_score(y_test_r, y_pred_r)

print('=== Acceptance Predictor Results ===')
print(f'RMSE:  {rmse:.6f}')
print(f'MAE:   {mae:.6f}')
print(f'R²:    {r2:.4f}')

In [ ]:
# Cross-validation R²
cv_r2 = cross_val_score(rf_reg, X_reg, y_reg, cv=5, scoring='r2')
print(f'5-Fold CV R²: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Scatter
ax = axes[0]
ax.scatter(y_test_r * 100, y_pred_r * 100, alpha=0.7, edgecolor='white', s=60, c='#3498db')
ax.plot([y_test_r.min()*100, y_test_r.max()*100], [y_test_r.min()*100, y_test_r.max()*100],
        'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Driver Quote Acceptance Rate (%)', fontsize=12)
ax.set_ylabel('Predicted (%)', fontsize=12)
ax.set_title('Predicted vs Actual — Acceptance Rate', fontsize=14, fontweight='bold')
ax.legend()

# Residuals
ax = axes[1]
residuals = (y_test_r - y_pred_r) * 100
ax.scatter(y_pred_r * 100, residuals, alpha=0.7, edgecolor='white', s=60, c='#e74c3c')
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Predicted (%)', fontsize=12)
ax.set_ylabel('Residual (%)', fontsize=12)
ax.set_title('Residual Plot', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 2.1 SHAP — Regressor Explainability

In [ ]:
explainer_reg = shap.TreeExplainer(rf_reg)
shap_values_reg = explainer_reg.shap_values(X_test_r)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

plt.sca(axes[0])
shap.summary_plot(shap_values_reg, X_test_r, plot_type='bar', show=False, max_display=10)
axes[0].set_title('SHAP Feature Importance — Acceptance Predictor', fontsize=12, fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(shap_values_reg, X_test_r, show=False, max_display=10)
axes[1].set_title('SHAP Beeswarm — Acceptance Predictor', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from RF
feat_imp = pd.DataFrame({
    'Feature': reg_features,
    'Importance': rf_reg.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='#2ecc71', edgecolor='white')
ax.set_title('Feature Importance — Acceptance Predictor', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Save regressor
joblib.dump(rf_reg, 'models/acceptance_predictor.pkl')
print('✅ Saved: models/acceptance_predictor.pkl')

---
## Model 3: Driver Behavior Clustering (KMeans, k=4)

**Features:** `driver_cancellation_rate`, `driver_quote_acceptance_rate`, `earnings_per_km`, `avg_distance`

**Evaluation:** Silhouette Score, Inertia, Cluster Profiles  
**Output:** Business-labeled clusters

In [ ]:
cluster_features = ['driver_cancellation_rate', 'driver_quote_acceptance_rate', 'earnings_per_km', 'avg_distance']

X_cluster = df[cluster_features].copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f'Clustering input shape: {X_scaled.shape}')

In [ ]:
# Elbow plot + Silhouette for validation
K_range = range(2, 9)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(4, color='red', linestyle='--', alpha=0.7, label='k=4')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].legend()

axes[1].plot(list(K_range), sil_scores, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(4, color='red', linestyle='--', alpha=0.7, label='k=4')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final KMeans with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df['cluster'])
inertia = kmeans.inertia_

print(f'Silhouette Score: {sil:.4f}')
print(f'Inertia: {inertia:.2f}')
print(f'\nCluster distribution:')
print(df['cluster'].value_counts().sort_index())

### 3.1 Cluster Profiles

In [ ]:
# Compute cluster profiles
profile_cols = cluster_features + ['conversion_rate', 'reliability_score', 'driver_earnings', 'completed_trips']
cluster_profiles = df.groupby('cluster')[profile_cols].agg(['mean', 'median', 'std']).round(4)
cluster_profiles

In [ ]:
# Simpler view with means
cluster_means = df.groupby('cluster')[cluster_features].mean().round(4)
print('=== Cluster Mean Profiles ===')
display(cluster_means)

In [ ]:
# Assign business labels based on cluster profiles
# We'll analyze the means and assign descriptive names

cluster_summary = df.groupby('cluster')[cluster_features].mean()

# Label assignment logic:
# High acceptance, low cancel → "⭐ Reliable Performers"
# Low acceptance, high cancel → "⚠️ High-Risk Zones"
# High earnings/km, long distance → "🚀 Premium Long-Haul"
# Moderate everything → "🔄 Average Balanced"

label_map = {}
for cluster_id in range(4):
    row = cluster_summary.loc[cluster_id]
    cancel = row['driver_cancellation_rate']
    accept = row['driver_quote_acceptance_rate']
    epk = row['earnings_per_km']
    dist = row['avg_distance']
    
    # Score-based assignment
    scores = {
        '⭐ Reliable Performers': accept * 2 - cancel * 2 - dist * 0.01,
        '⚠️ High-Risk Zones': cancel * 3 - accept * 2,
        '🚀 Premium Long-Haul': epk * 0.1 + dist * 0.1 - cancel,
        '🔄 Average Balanced': 0  # default
    }

assigned_labels = [None] * 4
available_labels = ['⭐ Reliable Performers', '⚠️ High-Risk Zones', '🚀 Premium Long-Haul', '🔄 Average Balanced']

# Sort clusters by characteristics and assign
# Best acceptance (highest) → Reliable
sorted_by_accept = cluster_summary['driver_quote_acceptance_rate'].sort_values(ascending=False).index.tolist()
# Worst cancel (highest) → High Risk
sorted_by_cancel = cluster_summary['driver_cancellation_rate'].sort_values(ascending=False).index.tolist()
# Longest distance → Premium Long Haul
sorted_by_dist = cluster_summary['avg_distance'].sort_values(ascending=False).index.tolist()

used = set()

# Assign Reliable: highest acceptance rate
for c in sorted_by_accept:
    if c not in used:
        assigned_labels[c] = '⭐ Reliable Performers'
        used.add(c)
        break

# Assign High Risk: highest cancel rate (among remaining)
for c in sorted_by_cancel:
    if c not in used:
        assigned_labels[c] = '⚠️ High-Risk Zones'
        used.add(c)
        break

# Assign Premium Long-Haul: longest avg distance (among remaining)
for c in sorted_by_dist:
    if c not in used:
        assigned_labels[c] = '🚀 Premium Long-Haul'
        used.add(c)
        break

# Remaining → Average Balanced
for c in range(4):
    if c not in used:
        assigned_labels[c] = '🔄 Average Balanced'
        used.add(c)

label_map = {i: assigned_labels[i] for i in range(4)}
df['cluster_label'] = df['cluster'].map(label_map)

print('=== Business Labels ===')
for k, v in label_map.items():
    count = (df['cluster'] == k).sum()
    print(f'  Cluster {k}: {v} ({count} wards)')

print(f'\n{df["cluster_label"].value_counts()}')

In [ ]:
# Detailed cluster profile table
print('=== Detailed Cluster Profiles ===')
detailed = df.groupby('cluster_label')[cluster_features + ['conversion_rate', 'reliability_score']].mean().round(4)
detailed

### 3.2 Cluster Visualizations

In [ ]:
# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
palette = {'⭐ Reliable Performers': '#2ecc71', '⚠️ High-Risk Zones': '#e74c3c',
           '🚀 Premium Long-Haul': '#3498db', '🔄 Average Balanced': '#f39c12'}

# Cancel Rate vs Acceptance Rate
sns.scatterplot(
    data=df, x='driver_cancellation_rate', y='driver_quote_acceptance_rate',
    hue='cluster_label', palette=palette, s=70, alpha=0.8, edgecolor='white', ax=axes[0]
)
axes[0].set_title('Cancel Rate vs Acceptance Rate', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8, loc='best')

# Earnings/km vs Distance
sns.scatterplot(
    data=df, x='avg_distance', y='earnings_per_km',
    hue='cluster_label', palette=palette, s=70, alpha=0.8, edgecolor='white', ax=axes[1]
)
axes[1].set_title('Avg Distance vs Earnings/km', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=8, loc='best')

# Cancel Rate vs Earnings/km
sns.scatterplot(
    data=df, x='driver_cancellation_rate', y='earnings_per_km',
    hue='cluster_label', palette=palette, s=70, alpha=0.8, edgecolor='white', ax=axes[2]
)
axes[2].set_title('Cancel Rate vs Earnings/km', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=8, loc='best')

plt.suptitle('Driver Behavior Clusters', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Radar chart for cluster profiles
from matplotlib.patches import FancyBboxPatch

categories = ['Cancel\nRate', 'Acceptance\nRate', 'Earnings\n/km', 'Avg\nDistance']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Normalize cluster means to [0, 1] for radar
cluster_means_norm = df.groupby('cluster_label')[cluster_features].mean()
for col in cluster_features:
    cmin, cmax = cluster_means_norm[col].min(), cluster_means_norm[col].max()
    if cmax > cmin:
        cluster_means_norm[col] = (cluster_means_norm[col] - cmin) / (cmax - cmin)
    else:
        cluster_means_norm[col] = 0.5

for label, row in cluster_means_norm.iterrows():
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=palette[label])
    ax.fill(angles, values, alpha=0.15, color=palette[label])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_title('Cluster Profiles — Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Save clustering artifacts
joblib.dump(kmeans, 'models/driver_behavior_kmeans.pkl')
joblib.dump(scaler, 'models/cluster_scaler.pkl')
joblib.dump(label_map, 'models/cluster_label_map.pkl')

# Save cluster assignments
df[['ward', 'cluster', 'cluster_label']].to_csv('models/ward_cluster_labels.csv', index=False)

print('✅ Saved: models/driver_behavior_kmeans.pkl')
print('✅ Saved: models/cluster_scaler.pkl')
print('✅ Saved: models/cluster_label_map.pkl')
print('✅ Saved: models/ward_cluster_labels.csv')

---
## Summary of All Models

In [ ]:
print('=' * 70)
print('                   DriverPulse — Model Summary')
print('=' * 70)

print('\n📊 Model 1: Ward Risk Classifier (Random Forest)')
print(f'   Accuracy:       {accuracy_score(y_test_c, y_pred_c):.4f}')
print(f'   F1 (macro):     {f1_score(y_test_c, y_pred_c, average="macro"):.4f}')
print(f'   F1 (weighted):  {f1_score(y_test_c, y_pred_c, average="weighted"):.4f}')
print(f'   CV F1 (macro):  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'   → models/ward_risk_classifier.pkl')

print(f'\n📈 Model 2: Acceptance Predictor (Random Forest Regressor)')
print(f'   RMSE:           {rmse:.6f}')
print(f'   MAE:            {mae:.6f}')
print(f'   R²:             {r2:.4f}')
print(f'   CV R²:          {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print(f'   → models/acceptance_predictor.pkl')

print(f'\n🔮 Model 3: Driver Behavior Clustering (KMeans k=4)')
print(f'   Silhouette:     {sil:.4f}')
print(f'   Inertia:        {inertia:.2f}')
print(f'   Clusters:')
for k, v in label_map.items():
    count = (df['cluster'] == k).sum()
    print(f'     {k}: {v} ({count} wards)')
print(f'   → models/driver_behavior_kmeans.pkl')

print('\n' + '=' * 70)
print('All models and artifacts saved to models/ ✅')
print('=' * 70)

In [ ]:
# List all saved files
import glob
saved_files = glob.glob('models/*')
for f in sorted(saved_files):
    size = os.path.getsize(f)
    print(f'  {f} ({size/1024:.1f} KB)')